In [112]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
mushroom = fetch_ucirepo(id=73) 
  
# data (as pandas dataframes) 
X = mushroom.data.features 
y = mushroom.data.targets.squeeze() 
print(mushroom.metadata) 
  
# variable information 
print(mushroom.variables) 

  

{'uci_id': 73, 'name': 'Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/73/mushroom', 'data_url': 'https://archive.ics.uci.edu/static/public/73/data.csv', 'abstract': 'From Audobon Society Field Guide; mushrooms described in terms of physical characteristics; classification: poisonous or edible', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 8124, 'num_features': 22, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['poisonous'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1981, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5959T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family (pp. 500-525).  Each species is identified as definitely edible, definitely po

In [107]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
import numpy as np
import pandas as pd

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


# Problem 4: Naive Bayes classifier (25 points)

In this problem you will implement your own Naive Bayes classifier and you will compare it with a package implementation. You will use the
Mushroom dataset for this problem. Split the dataset into 75% for training and 25% for testing.

1. Train the Naive Bayes classifier. Compute the prior probabilities for the `*Edible*` and `*Poisonous*` classes from the training data. 

For each feature $X_i$ in the dataset compute the probabilities $P[X_i = x| Y=\textit{Edible}]$, and $P[X_i = x| Y=\textit{Poisonous}]$ from the training data. Use the Laplace smoothing method when computing these probabilities. Note that the Naive Bayes classifier stores these prior and conditional probabilities.

In [ ]:
def fit_naive_bayes(X_train, y_train):
    classes = y_train.unique()
    priors = {c: np.log((y_train == c).sum() / len(y_train)) for c in classes}
    cond = {}
    for c in classes:
        subset = X_train[y_train == c]  # rows belonging to class c
        cond[c] = {}
        for col in X_train.columns:
            vals = X_train[col].unique()
            k = len(vals)                        # number of unique values (for Laplace smoothing)
            counts = subset[col].value_counts()
            n_c = len(subset)
            # log P(X_i=v | Y=c) with Laplace smoothing: (count + 1) / (n_c + k)
            cond[c][col] = {v: np.log((counts.get(v, 0) + 1) / (n_c + k)) for v in vals}
    return priors, cond

priors, cond = fit_naive_bayes(X_train, y_train)

# prioe probabilities
print("Prior probabilities:")
for c, logp in priors.items():
    print(f"{c}: {np.exp(logp):.4f}") 

# conditional probabilities for all features
for col in X_train.columns:
    print(f"\nFeature: {col}")
    for c in ['e', 'p']:  # Edible, Poisonous
        print(f"  Class {c}:")
        for val, logprob in cond[c][col].items():
            print(f"    P({col}={val} | {c}) = {np.exp(logprob):.4f}")

def predict_naive_bayes(X_test, priors, cond):
    classes = list(priors.keys())
    preds = []
    for _, row in X_test.iterrows():
        scores = {}
        for c in classes:
            # log P(Y=c) + sum of log P(Xi=xi | Y=c) across all features
            # unseen values fall back to 0 (i.e. log(1), a neutral contribution)
            scores[c] = priors[c] + sum(cond[c][col].get(row[col], 0) for col in X_test.columns)
        preds.append(max(scores, key=scores.get))  # predict class with highest log-posterior
    return np.array(preds)
y_pred_custom = predict_naive_bayes(X_test, priors, cond)



Prior probabilities:
p: 0.4820
e: 0.5180

Feature: cap-shape
  Class e:
    P(cap-shape=x | e) = 0.4684
    P(cap-shape=k | e) = 0.0547
    P(cap-shape=f | e) = 0.3751
    P(cap-shape=b | e) = 0.0936
    P(cap-shape=s | e) = 0.0079
    P(cap-shape=c | e) = 0.0003
  Class p:
    P(cap-shape=x | p) = 0.4410
    P(cap-shape=k | p) = 0.1485
    P(cap-shape=f | p) = 0.3955
    P(cap-shape=b | p) = 0.0129
    P(cap-shape=s | p) = 0.0003
    P(cap-shape=c | p) = 0.0017

Feature: cap-surface
  Class e:
    P(cap-surface=s | e) = 0.2766
    P(cap-surface=f | e) = 0.3722
    P(cap-surface=y | e) = 0.3509
    P(cap-surface=g | e) = 0.0003
  Class p:
    P(cap-surface=s | p) = 0.3611
    P(cap-surface=f | p) = 0.2016
    P(cap-surface=y | p) = 0.4359
    P(cap-surface=g | p) = 0.0014

Feature: cap-color
  Class e:
    P(cap-color=e | e) = 0.1459
    P(cap-color=y | e) = 0.0976
    P(cap-color=n | e) = 0.3023
    P(cap-color=w | e) = 0.1646
    P(cap-color=g | e) = 0.2486
    P(cap-color=p | e) = 0

2. For each point in the testing set estimate the probability that it belongs to the `*Edible*` and `*Poisonous*` classes. Use the Naive Bayes
classifier probabilities computed in part (1).

In [109]:

def predict_naive_bayes_proba(X_test, priors, cond):
    """
    Returns posterior probabilities P(Y=c | X) for each test row.
    """
    classes = list(priors.keys())
    prob_rows = []

    for _, row in X_test.iterrows():
        # Compute log-scores
        log_scores = {}
        for c in classes:
            log_scores[c] = priors[c] + sum(cond[c][col].get(row[col], 0) for col in X_test.columns)
        
        # Convert log-scores to probabilities
        max_log = max(log_scores.values()) 
        exp_scores = {c: np.exp(log_scores[c] - max_log) for c in classes}  # subtract max_log
        total = sum(exp_scores.values())
        probs = {c: exp_scores[c] / total for c in classes}  # normalized
        prob_rows.append(probs)

    return prob_rows

probs = predict_naive_bayes_proba(X_test, priors, cond)
for i, p in enumerate(probs[:5]):  # first 5 rows
    print(f"Test row {i+1}:")
    print(f"  P(Edible) = {p['e']:.4f}")
    print(f"  P(Poisonous) = {p['p']:.4f}\n")

Test row 1:
  P(Edible) = 1.0000
  P(Poisonous) = 0.0000

Test row 2:
  P(Edible) = 0.0000
  P(Poisonous) = 1.0000

Test row 3:
  P(Edible) = 1.0000
  P(Poisonous) = 0.0000

Test row 4:
  P(Edible) = 0.9795
  P(Poisonous) = 0.0205

Test row 5:
  P(Edible) = 0.0000
  P(Poisonous) = 1.0000



3. Compute accuracy, precision, recall, and F1 score for your Naive Bayes classifier on the testing data.

In [110]:
def evaluate(y_true, y_pred, pos_label='p'):
    # pos_label='p' means Poisonous is the positive class (safety-critical to detect)
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=pos_label),
        "Recall":    recall_score(y_true, y_pred, pos_label=pos_label),
        "F1":        f1_score(y_true, y_pred, pos_label=pos_label),
    }

custom_metrics = evaluate(y_test, y_pred_custom)
print("Custom NB:", custom_metrics)


Custom NB: {'Accuracy': 0.9527326440177253, 'Precision': 0.9911012235817576, 'Recall': 0.9101123595505618, 'F1': 0.9488817891373802}


4. Compare the results obtained by your implementation with those obtained with a Naive Bayes package (trained on the same dataset).
Use several metrics, including accuracy, precision, recall, and F1 score. Are the results similar or different?

In [111]:
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_enc = enc.fit_transform(X_train.fillna('missing'))  # fillna so NaNs become their own category
X_test_enc  = enc.transform(X_test.fillna('missing'))
skl_nb = CategoricalNB(alpha=1.0).fit(X_train_enc, y_train)  # alpha=1.0 is Laplace smoothing

y_pred_skl = skl_nb.predict(X_test_enc)
skl_metrics = evaluate(y_test, y_pred_skl)

print(f"\n{'Metric':<12} {'Custom NB':>10} {'sklearn NB':>10}")
for m in custom_metrics:
    print(f"{m:<12} {custom_metrics[m]:>10.4f} {skl_metrics[m]:>10.4f}")


Metric        Custom NB sklearn NB
Accuracy         0.9527     0.9527
Precision        0.9911     0.9911
Recall           0.9101     0.9101
F1               0.9489     0.9489


The results are the same for both custom and package Naive Bayes implementation